# 03 - QP Interface With `qpsol`

This notebook introduces CasADi's high-level QP interface.

Learning goals:

- Write QPs in CasADi's high-level form.
- Use `x0`, `lbx`, `ubx`, `lbg`, and `ubg`.
- Add parameters through the `p` field.
- Solve least-squares QPs that will later become inverse-kinematics steps.

In [ ]:
# Colab setup.
# These tutorials assume a fresh Google Colab runtime.
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "casadi",
    "numpy",
    "matplotlib",

])


In [1]:
import casadi as ca
import numpy as np

print("CasADi version:", ca.__version__)
qp_opts = {"printLevel": "none", "print_time": False}

CasADi version: 3.7.2


## The `qpsol` Solver Object Contract

A CasADi QP solver is created once from a symbolic problem description. The solver object is then called many times with numeric data.

The construction-time pattern is

```python
qp = {
    "x": x,  # decision variables, required
    "f": f,  # scalar quadratic objective, required
    "g": g,  # constraint expression, optional
    "p": p,  # parameters, optional
}
solver = ca.qpsol("solver_name", "qpoases", qp, opts)
```

The corresponding optimization template is

$$
\begin{aligned}
\min_x \quad & f(x, p) \\
\text{s.t.} \quad
& x_{\min} \le x \le x_{\max}, \\
& g_{\min} \le g(x, p) \le g_{\max}.
\end{aligned}
$$

### Construction-Time Pieces

| Piece | Meaning |
|---|---|
| `"x"` | Symbolic decision variable vector. |
| `"f"` | Scalar quadratic objective expression. |
| `"g"` | Optional vector of equality or inequality constraint expressions. |
| `"p"` | Optional symbolic parameter vector. Parameters are fixed during one solve. |
| solver plugin | Here we use `"qpoases"`. |
| `opts` | Construction-time solver options, for example `{"printLevel": "none", "print_time": False}`. |

### Call-Time Inputs

A solver is a CasADi `Function`, so it is called with named inputs:

```python
sol = solver(
    x0=x0_value,
    p=p_value,
    lbx=x_lower,
    ubx=x_upper,
    lbg=g_lower,
    ubg=g_upper,
    lam_x0=lam_x_guess,
    lam_g0=lam_g_guess,
)
```

| Input | Meaning |
|---|---|
| `x0` | Initial guess or warm start for `x`. |
| `p` | Numeric parameter values matching the symbolic `p`. |
| `lbx`, `ubx` | Lower and upper bounds on decision variables. |
| `lbg`, `ubg` | Lower and upper bounds on constraint expressions `g`. Use equal values for equality constraints. |
| `lam_x0`, `lam_g0` | Optional warm starts for variable-bound and constraint multipliers. |

### Outputs

The result dictionary contains:

| Output | Meaning |
|---|---|
| `x` | Optimizer. |
| `f` | Objective value at the optimizer. |
| `g` | Constraint values at the optimizer. |
| `lam_x`, `lam_g`, `lam_p` | Multipliers/sensitivities returned by the solver. |

The examples below are all special cases of this same template.

## Example 1: Unconstrained QP

Decision variable:

$$
x = \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}.
$$

Problem:

$$
\begin{aligned}
\min_{x \in \mathbb{R}^2} \quad
& \frac{1}{2}\left((x_0 - 1)^2 + 2(x_1 + 2)^2\right).
\end{aligned}
$$

In [ ]:
x = ca.SX.sym("x", 2)
f = 0.5 * ((x[0] - 1.0)**2 + 2.0 * (x[1] + 2.0)**2)

qp = {"x": x, "f": f}
S = ca.qpsol("unconstrained_qp", "qpoases", qp, qp_opts)  # Declare once the Solver and then recall it!
sol = S(x0=[0.0, 0.0])

print("x* =", sol["x"])
print("f* =", sol["f"])


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.

x* = [1, -2]
f* = -8.88178e-16


## Example 2: Box Constraints

Use the same objective as Example 1, now with hard bounds on the decision variable:

$$
\begin{aligned}
\min_{x \in \mathbb{R}^2} \quad
& \frac{1}{2}\left((x_0 - 1)^2 + 2(x_1 + 2)^2\right) \\
\text{s.t.} \quad
& 0 \le x_0 \le 2, \\
& -1 \le x_1 \le 2.
\end{aligned}
$$

In [3]:
sol_box = S(
    x0=[0.0, 0.0],
    lbx=[0.0, -1.0],
    ubx=[2.0, 2.0],
)

print("box-constrained x* =", sol_box["x"])
print("box-constrained f* =", sol_box["f"])

box-constrained x* = [1, -1]
box-constrained f* = 1


## Example 3: Linear Inequality Constraints

Decision variable:

$$
x = \begin{bmatrix} x_0 \\ x_1 \end{bmatrix}.
$$

Problem:

$$
\begin{aligned}
\min_{x \in \mathbb{R}^2} \quad
& \frac{1}{2} x^T x \\
\text{s.t.} \quad
& x_0 + x_1 \ge 1.
\end{aligned}
$$

Use `g` for the left-hand side and bind it with `lbg` and `ubg`.

In [4]:
x = ca.SX.sym("x", 2)
f = 0.5 * ca.dot(x, x)
g = ca.vertcat(x[0] + x[1])

qp = {"x": x, "f": f, "g": g}
S_linear = ca.qpsol("linear_constraint_qp", "qpoases", qp, qp_opts)

sol_linear = S_linear(
    x0=[0.0, 0.0],
    lbg=[1.0],
    ubg=[ca.inf],
)

print("x* =", sol_linear["x"])
print("constraint value g(x*) =", sol_linear["g"])


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.

x* = [0.5, 0.5]
constraint value g(x*) =

## Example 4: Parametric Least-Squares QP

This is the pattern used in the QP inverse-kinematics notebook. The data matrix `J` and target residual `e` are parameters, while the decision variable is `dq`.

Decision variable:

$$
\Delta q \in \mathbb{R}^2.
$$

Parameters:

$$
J \in \mathbb{R}^{3 \times 2}, \qquad e \in \mathbb{R}^3.
$$

Problem:

$$
\begin{aligned}
\min_{\Delta q \in \mathbb{R}^2} \quad
& \frac{1}{2}\|J \Delta q - e\|_2^2
+ \frac{\lambda}{2}\|\Delta q\|_2^2 \\
\text{s.t.} \quad
& -0.3 \le \Delta q_i \le 0.3, \quad i=0,1.
\end{aligned}
$$

In [ ]:
n = 2
m = 3

dq = ca.SX.sym("dq", n)
J_param = ca.SX.sym("J", m, n)
e_param = ca.SX.sym("e", m)
p = ca.vertcat(ca.reshape(J_param, m * n, 1), e_param) # trafo matrix into m*n x 1 Vector: 6 x 1 and below e

damping = 1e-6
objective = 0.5 * ca.sumsqr(J_param @ dq - e_param) + 0.5 * damping * ca.sumsqr(dq)

qp = {"x": dq, "p": p, "f": objective}
S_ls = ca.qpsol("least_squares_qp", "qpoases", qp, qp_opts)

J_num = ca.DM([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
e_num = ca.DM([0.4, -0.2, 0.1])
p_num = ca.vertcat(ca.reshape(J_num, m * n, 1), e_num)

sol_ls = S_ls(
    x0=[0.0, 0.0],
    p=p_num,
    lbx=[-0.3, -0.3],
    ubx=[0.3, 0.3],
)

print("dq* =", sol_ls["x"])
print("J dq* =", J_num @ sol_ls["x"])
print("target e =", e_num)

## Exercise

Modify the least-squares QP so that the sum of the decision variables is upper bounded.

Problem:

$$
\begin{aligned}
\min_{\Delta q \in \mathbb{R}^2} \quad
& \frac{1}{2}\|J \Delta q - e\|_2^2
+ \frac{\lambda}{2}\|\Delta q\|_2^2 \\
\text{s.t.} \quad
& -0.3 \le \Delta q_i \le 0.3, \quad i=0,1, \\
& \Delta q_0 + \Delta q_1 \le 0.1.
\end{aligned}
$$

Use `g`, `lbg`, and `ubg` for the final linear inequality.

In [6]:
n = 2
m = 3

dq = ca.SX.sym("dq", n)
J_param = ca.SX.sym("J", m, n)
e_param = ca.SX.sym("e", m)
p = ca.vertcat(ca.reshape(J_param, m * n, 1), e_param) # trafo matrix into m*n x 1 Vector: 6 x 1 and below e

damping = 1e-6
objective = 0.5 * ca.sumsqr(J_param @ dq - e_param) + 0.5 * damping * ca.sumsqr(dq)
g = dq[0] + dq[1]

qp = {"x": dq, "p": p, "f": objective, "g":g}
S_ls = ca.qpsol("least_squares_qp", "qpoases", qp, qp_opts)

J_num = ca.DM([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
e_num = ca.DM([0.4, -0.2, 0.1])
p_num = ca.vertcat(ca.reshape(J_num, m * n, 1), e_num)

sol_ls = S_ls(
    x0=[0.0, 0.0],
    p=p_num,
    lbx=[-0.3, -0.3],
    ubx=[0.3, 0.3],
    lbg=[-ca.inf],
    ubg=[0.1]
)

print("dq* =", sol_ls["x"])
print("J dq* =", J_num @ sol_ls["x"])
print("target e =", e_num)


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.


qpOASES -- An Implementation of the Online Active Set Strategy.
Copyright (C) 2007-2015 by Hans Joachim Ferreau, Andreas Potschka,
Christian Kirches et al. All rights reserved.

qpOASES is distributed under the terms of the 
GNU Lesser General Public License 2.1 in the hope that it will be 
useful, but WITHOUT ANY WARRANTY; without even the implied warranty 
of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. 
See the GNU Lesser General Public License for more details.

dq* = [0.3, -0.2]
J dq* = [0.3, -0.2, 0.